# **프로젝트 미션 — "어디가 비싸고, 왜 비싼가?"**

## 2차시 보너스 · NYC Airbnb 6단계 시각적 탐색

---

> **본 미션의 목표**
>
> 본 차시에서 배운 시각화 도구를 **실전 데이터**(NYC Airbnb 2019)에 적용하여, **6단계의 시각적 탐색**으로 가격의 패턴을 증명한다. 본 미션은 **단순 도구 사용**이 아니라, **수치 기반의 관찰 보고서**를 작성하는 훈련이다.

## **미션 규칙**

각 단계마다 그래프를 그리고, **관찰 노트**에 발견한 내용을 기록한다. 다음 4가지 규칙을 엄격히 준수한다.

| 규칙 | 내용 |
|---|---|
| **수치 포함** | 모든 관찰에 구체적 수치(`$`, `%`, `배수`)를 반드시 포함한다 |
| **색맹 안전** | 모든 그래프에 `colorblind`, `viridis`, `cividis` 팔레트를 사용한다 |
| **상관 vs 인과** | "~때문이다"(인과) 표현 대신 "~와 관련이 있다"(상관)로 표현한다 |
| **그래프 근거** | 그래프에 보이지 않는 추측은 작성하지 않는다 |

## **합격 vs 불합격 예시**

```
❌ 불합격: "Manhattan이 비싸다."
   → 일반론. 어떤 수치로 비싼지 명시되지 않음.

✓ 합격: "Manhattan의 중앙값($150)은 Bronx($65)의 약 2.3배이며,
        IQR도 $95로 가장 넓어 가격 편차가 크다."
   → 구체적 수치($150, $65, 2.3배, $95)와 IQR 정보까지 포함.
```

## **6단계 흐름도**

```
1단계 → 2단계 → 3단계 → 4단계 → 5단계 → 6단계
 분포   지역    위치    유형     상관    종합
       비교    지도    교차    분석    관찰
       
히스토   박스    Plotly   히트맵   히트맵   미니
+KDE    바이올린 산점도   교차표  +regplot 보고서
```


---

## **0. 환경 설정**(Setup)

다음 셀을 실행하여 한글 폰트, **색맹 안전 팔레트**, 고화질 출력을 한 번에 설정한다.

In [ ]:
# 환경 설정
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# 한글 폰트 (Colab)
if 'COLAB_GPU' in os.environ or os.path.exists('/content'):
    os.system('apt-get -qq install -y fonts-nanum > /dev/null 2>&1')
    import matplotlib.font_manager as fm
    font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    if os.path.exists(font_path):
        fm.fontManager.addfont(font_path)
        plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

# ★ 색맹 안전 팔레트
sns.set_palette('colorblind')
sns.set_style('whitegrid')

print('환경 설정 완료')

## **0.1 데이터 로드**(Load)

뉴욕 에어비앤비 2019년 데이터를 불러온다.

In [ ]:
# Airbnb NYC 2019 로드
URL = ('https://raw.githubusercontent.com/leina99-lab/classes/main/'
       'AI%ED%94%84%EB%A1%9C%EA%B7%B8%EB%9E%98%EB%B0%8D/data/AB_NYC_2019.csv')

try:
    df = pd.read_csv(URL)
    print(f'로드 성공: {df.shape}')
except Exception:
    # 폴백 합성 데이터
    np.random.seed(42)
    n = 1500
    df = pd.DataFrame({
        'name': [f'Listing {i}' for i in range(n)],
        'neighbourhood_group': np.random.choice(
            ['Manhattan','Brooklyn','Queens','Bronx','Staten Island'],
            n, p=[0.45,0.40,0.10,0.03,0.02]),
        'room_type': np.random.choice(
            ['Entire home/apt','Private room','Shared room'],
            n, p=[0.52,0.45,0.03]),
        'price': np.random.lognormal(4.7, 0.7, n).astype(int) + 30,
        'minimum_nights': np.random.choice([1,2,3,5,7,30], n,
                                           p=[0.40,0.20,0.10,0.10,0.10,0.10]),
        'number_of_reviews': np.random.poisson(15, n),
        'reviews_per_month': np.round(np.random.exponential(1.0, n), 2),
        'calculated_host_listings_count': np.random.choice([1,2,3,5,10,50], n,
                                           p=[0.65,0.15,0.08,0.05,0.04,0.03]),
        'availability_365': np.random.randint(0, 366, n),
        'longitude': np.random.uniform(-74.05, -73.85, n),
        'latitude': np.random.uniform(40.65, 40.85, n),
    })
    print(f'폴백 데이터 사용: {df.shape}')

# 가격 정제 ($10 미만 또는 $1000 초과 제거)
df_clean = df[df['price'].between(10, 1000)].copy()
df_under300 = df_clean[df_clean['price'] <= 300].copy()

print(f'정제 후: {df_clean.shape}')
print(f'$300 이하: {df_under300.shape}')
df_clean.head(3)

---

# **1단계 — 전체 가격 분포**

> **질문**: 뉴욕 전체 숙소의 가격은 어떻게 퍼져 있는가?

전체 분포를 한 번에 보면 이상치 때문에 상세 모양이 짜부라진다. **(A) 전체** + **(B) $300 이하 확대**의 **두 패널**로 비교한다.

In [ ]:
# 1단계 - (A) 전체, (B) $300 이하 확대
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (A) 전체 분포
sns.histplot(data=df_clean, x='price', bins=50, kde=True, ax=axes[0])
axes[0].set_title('(A) 전체 가격 분포 — 이상치 포함', fontsize=12, fontweight='bold')
axes[0].set_xlabel('가격 ($)'); axes[0].set_ylabel('빈도')

# (B) $300 이하 확대
sns.histplot(data=df_under300, x='price', bins=30, kde=True, ax=axes[1])
axes[1].set_title('(B) $300 이하 — 분포 모양이 잘 보임', fontsize=12, fontweight='bold')
axes[1].set_xlabel('가격 ($)'); axes[1].set_ylabel('빈도')

plt.tight_layout(); plt.show()

# 기술통계
print(f"평균:        ${df_clean['price'].mean():.0f}")
print(f"중앙값:      ${df_clean['price'].median():.0f}")
print(f"최빈값:      ${df_clean['price'].mode()[0]}")
print(f"왜도(skew): {df_clean['price'].skew():.2f}")

### **관찰 노트 1**

다음 빈칸을 코드 출력 결과를 보고 채우시오.

> 전체 가격 분포는 오른쪽으로 ____________(왜도 = ____________)이며, 봉우리(최빈값)는 약 \$____________에 위치한다. 평균(\$____________)이 중앙값(\$____________)보다 ____________(높/낮)아, 고가 숙소가 평균을 끌어올리고 있다.

> **단서**: 왜도 > 1이면 강하게 오른쪽 꼬리. 평균 > 중앙값은 오른쪽 꼬리 분포의 전형적 신호이다.

---

# **2단계 — 지역별 분포 비교 (박스플롯 + 바이올린)**

> **질문**: 어느 자치구가 비싸고, 가격 편차는 얼마나 큰가?

## **2-A. 박스플롯**(중앙값·IQR·이상치)

In [ ]:
# 2-A. 박스플롯 - 중앙값 내림차순 정렬
order = (df_clean.groupby('neighbourhood_group')['price']
         .median().sort_values(ascending=False).index)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# (A) 전체
sns.boxplot(data=df_clean, x='neighbourhood_group', y='price',
            order=order, palette='colorblind', ax=axes[0])
axes[0].set_title('(A) 지역별 가격 — 이상치 포함', fontsize=12, fontweight='bold')
axes[0].set_xlabel('자치구'); axes[0].set_ylabel('가격 ($)')

# (B) $300 이하
sns.boxplot(data=df_under300, x='neighbourhood_group', y='price',
            order=order, palette='colorblind', ax=axes[1])
axes[1].set_title('(B) $300 이하 — 박스 내부가 잘 보임', fontsize=12, fontweight='bold')
axes[1].set_xlabel('자치구'); axes[1].set_ylabel('가격 ($)')

plt.tight_layout(); plt.show()

# 자치구별 통계
stats = (df_clean.groupby('neighbourhood_group')['price']
         .agg(['median','mean','std','count'])
         .sort_values('median', ascending=False).round(0))
print('\n자치구별 가격 통계:')
print(stats)

## **2-B. 바이올린 플롯**(분포 모양까지)

박스플롯이 놓치는 **봉우리 개수**(이봉분포 등)를 바이올린이 보여준다.

In [ ]:
# 2-B. 바이올린 플롯
plt.figure(figsize=(12, 6))
sns.violinplot(data=df_under300, x='neighbourhood_group', y='price',
               order=order, palette='colorblind',
               inner='quartile',  # 사분위선 표시
               cut=0)             # 데이터 범위 밖 곡선 제거
plt.title('자치구별 가격 바이올린 플롯 ($300 이하)',
          fontsize=13, fontweight='bold')
plt.xlabel('자치구'); plt.ylabel('가격 ($)')
plt.show()

### **관찰 노트 2**

> 중앙값이 가장 높은 자치구는 ____________(중앙값 \$____________), 가장 낮은 자치구는 ____________(중앙값 \$____________)이다. 두 자치구의 중앙값 차이는 약 ____________배이다.

> IQR(상자 길이)이 가장 넓은 자치구는 ____________로, 해당 자치구의 가격 ____________이(가) 크다.

> 바이올린에서 ____________ 자치구의 봉우리는 ____________개이며, 이는 가격대가 ____________ 그룹으로 나뉘어 있음을 시사한다.

---

# **3단계 — 위치 × 가격 지도 (산점도)**

> **질문**: 비싼 숙소는 지리적으로 어디에 모여 있는가?

## **3-A. Matplotlib 정적 산점도**

위도/경도 좌표를 그대로 사용하면 NYC 지도 위에 점을 찍는 효과가 난다. 색상으로 가격을 표시한다.

In [ ]:
# 3-A. 정적 산점도 - 위치 × 가격
plt.figure(figsize=(10, 10))
scatter = plt.scatter(
    df_clean['longitude'], df_clean['latitude'],
    c=df_clean['price'],
    cmap='viridis',  # ★ 색맹 안전 연속 컬러맵
    alpha=0.3, s=5
)
plt.colorbar(scatter, label='가격 ($)')
plt.title('NYC Airbnb — 위치별 가격 (밝은 노란색일수록 고가)',
          fontsize=13, fontweight='bold')
plt.xlabel('경도(Longitude)')
plt.ylabel('위도(Latitude)')
plt.show()

## **3-B. Plotly 인터랙티브 산점도**

마우스 hover로 개별 숙소의 정보를 직접 확인한다.

In [ ]:
# 3-B. Plotly 인터랙티브
import plotly.express as px

fig = px.scatter(
    df_clean,
    x='longitude', y='latitude',
    color='price',
    color_continuous_scale='viridis',  # ★ 색맹 안전
    hover_data={'name': True,
                'neighbourhood_group': True,
                'room_type': True,
                'price': ':.0f',
                'longitude': ':.3f',
                'latitude': ':.3f'},
    opacity=0.4,
    title='NYC Airbnb — 인터랙티브 가격 지도',
    labels={'longitude':'경도', 'latitude':'위도', 'price':'가격($)'},
)
fig.update_layout(width=850, height=700)
fig.show()

### **탐색 과제 — Plotly로 직접 해보기**

1. **Manhattan 미드타운**(중심부)을 드래그로 확대하라.
2. **밝은 노란색**(고가) 점을 hover로 찾아 숙소 이름과 가격을 확인하라.
3. **Brooklyn Williamsburg** 근처에 고가 숙소가 있는지 살펴보라.
4. 가장 비싼 숙소 1개의 정보를 기록하라.

### **관찰 노트 3**

> 인터랙티브 지도 탐색 결과, 고가 숙소가 가장 밀집된 지역은 ____________ 이다. 해당 지역의 대표적 고가 숙소는 "____________" (room_type: ____________, \$____________ /박)이다.

> Brooklyn에서도 ____________ 지역에 고가 숙소가 분포하며, 이는 Manhattan 인접성과 ____________ 관련이 있어 보인다.

---

# **4단계 — room_type × 지역 교차 (히트맵)**

> **질문**: 같은 자치구라도 숙소 유형에 따라 가격이 어떻게 다른가?

## **4-A. 중앙 가격 히트맵**

In [ ]:
# 4-A. 중앙 가격 교차표 + 히트맵
price_cross = pd.crosstab(
    index=df_clean['neighbourhood_group'],
    columns=df_clean['room_type'],
    values=df_clean['price'],
    aggfunc='median'
).round(0)

plt.figure(figsize=(8, 5))
sns.heatmap(price_cross, annot=True, fmt='.0f',
            cmap='viridis',  # ★ 색맹 안전 연속 컬러맵
            linewidths=0.5,
            cbar_kws={'label':'중앙 가격($)'})
plt.title('자치구 × 숙소 유형별 중앙 가격', fontsize=13, fontweight='bold')
plt.ylabel('자치구'); plt.xlabel('숙소 유형')
plt.tight_layout(); plt.show()

print('\n교차표:')
print(price_cross)

## **4-B. 매물 수 히트맵**(비교용)

가격이 높다고 매물이 많은 것은 아니다. **매물 수**와 **가격**을 나란히 보면 시장 구조가 드러난다.

In [ ]:
# 4-B. 매물 수 + 중앙 가격 - 두 히트맵 나란히
count_cross = pd.crosstab(
    index=df_clean['neighbourhood_group'],
    columns=df_clean['room_type']
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# (A) 매물 수
sns.heatmap(count_cross, annot=True, fmt='d',
            cmap='cividis',  # ★ 색맹 전용 설계
            linewidths=0.5, ax=axes[0],
            cbar_kws={'label':'매물 수'})
axes[0].set_title('(A) 자치구 × 유형별 매물 수', fontsize=12, fontweight='bold')

# (B) 중앙 가격
sns.heatmap(price_cross, annot=True, fmt='.0f',
            cmap='viridis',  # ★ 색맹 안전
            linewidths=0.5, ax=axes[1],
            cbar_kws={'label':'중앙 가격($)'})
axes[1].set_title('(B) 자치구 × 유형별 중앙 가격', fontsize=12, fontweight='bold')

plt.tight_layout(); plt.show()

### **관찰 노트 4**

> 매물이 가장 많은 조합은 ____________ × ____________ (____________개)이고, 중앙 가격이 가장 높은 조합은 ____________ × ____________ (\$____________)이다. 두 조합은 ____________(같다 / 다르다).

> 같은 `Private room`이라도 자치구에 따라 중앙 가격이 \$____________(최저) ~ \$____________(최고)로 약 ____________ 배 차이가 난다.

---

# **5단계 — 숫자 변수 상관 분석**

> **질문**: 가격을 가장 잘 설명하는 숫자 변수는 무엇인가?

## **5-A. 상관 히트맵**(`coolwarm` + 삼각형 마스크)

In [ ]:
# 5-A. 상관 히트맵 - 하삼각만 표시
num_cols = ['price','minimum_nights','number_of_reviews',
            'reviews_per_month','calculated_host_listings_count',
            'availability_365']

corr = df_clean[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))  # 상삼각 가림

plt.figure(figsize=(9, 7))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm',  # ★ 빨강-파랑 = 색맹 안전 발산형
            center=0, vmin=-1, vmax=1,
            linewidths=0.5, square=True,
            cbar_kws={'label':'상관계수'})
plt.title('숫자 변수 간 상관계수 (하삼각)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## **5-B. 가격과 가장 상관 높은 변수의 회귀선**

In [ ]:
# 5-B. 가격과 상관 높은 상위 2개 변수의 regplot
price_corr = corr['price'].drop('price').abs().sort_values(ascending=False)
print('가격(price)과의 절대 상관계수:')
print(price_corr.round(3))

top2 = price_corr.index[:2]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, col in enumerate(top2):
    sns.regplot(data=df_under300, x=col, y='price',
                scatter_kws={'alpha':0.15, 's':10},
                line_kws={'color':'crimson', 'lw':2},
                ax=axes[i])
    r = df_under300[[col,'price']].corr().iloc[0,1]
    axes[i].set_title(f'price vs {col} (r = {r:.3f})',
                       fontsize=12, fontweight='bold')

plt.tight_layout(); plt.show()

### **관찰 노트 5**

> 가격(price)과 가장 높은 상관을 보이는 변수는 ____________ 이며, 상관계수는 약 r = ____________ 이다.

> 다만 \|r\| < 0.3이면 ____________ 상관이다. 따라서 에어비앤비 가격은 ____________ 단일 숫자 변수로는 충분히 설명되지 않으며, **위치(자치구)** 와 **숙소 유형** 같은 ____________ 변수가 더 중요한 것으로 보인다.

> 이는 본 차시에서 배운 원칙 "____________ ≠ ____________"의 좋은 예시이다.

---

# **6단계 — 종합 관찰 보고서**

> **질문**: 1~5단계의 발견을 한 보고서로 통합하면 무엇을 말할 수 있는가?

아래 양식에 **최소 3개**의 관찰을 작성한다. 각 관찰에는 **반드시 구체적인 수치**가 포함되어야 한다.

---

### **관찰 1 — 분포 (1단계 기반)**

> 전체 가격 분포는 ___________________________________________________________
>
> ___________________________________________________________________________

### **관찰 2 — 자치구 비교 (2단계 기반)**

> 자치구별로 가격 차이가 크다. 구체적으로 _________________________________________
>
> ___________________________________________________________________________

### **관찰 3 — 숙소 유형 (4단계 기반)**

> 같은 자치구라도 room_type에 따라 _____________________________________________
>
> ___________________________________________________________________________

### **(선택) 관찰 4 — 지리적 패턴 (3단계 기반)**

> 지도 시각화에서 발견한 패턴은 _______________________________________________
>
> ___________________________________________________________________________

### **(선택) 관찰 5 — 상관 (5단계 기반)**

> 숫자 변수 간 상관 분석 결과 _________________________________________________
>
> ___________________________________________________________________________

---

## **자기 점검 체크리스트**

다음 항목을 모두 만족해야 보고서가 완성된 것이다.

- [ ] 각 관찰에 구체적 수치(\$, %, 배수)가 들어 있는가?
- [ ] "~때문이다"(인과)가 아닌 "~와 관련이 있다"(상관)로 표현했는가?
- [ ] 그래프에 보이지 않는 추측을 작성하지 않았는가?
- [ ] 모든 그래프에 색맹 안전 팔레트(`colorblind`, `viridis`, `cividis`, `coolwarm`)를 사용했는가?
- [ ] 6단계를 모두 거쳤는가?

---

## **발표 가이드**(1분)

본 미션의 가장 흥미로운 관찰 **1개**를 선택하여 1분 발표를 준비한다.

| 순서 | 내용 | 시간 |
|---|---|---|
| 1 | 어떤 관찰을 했는지 한 문장 | 10초 |
| 2 | 근거 그래프를 화면에 띄움 | 5초 |
| 3 | 그래프에서 보이는 패턴을 구체적 수치와 함께 설명 | 35초 |
| 4 | 마무리 한 문장 | 10초 |

### **발표 모범 예시**

> "저는 **Manhattan과 Bronx의 가격 차이**를 발견했습니다.
> [박스플롯을 띄움]
> 이 박스플롯을 보면, Manhattan의 중앙값은 \$150이고 Bronx는 \$65입니다. 약 2.3배 차이입니다.
> 또한 Manhattan의 IQR(상자 길이)은 \$95로 가장 넓어, 같은 자치구 안에서도 가격 다양성이 가장 큽니다.
> 즉, **Manhattan은 비쌀 뿐 아니라 편차도 큰 자치구**입니다."

---

## **추가 도전 과제**(시간이 남는다면)

다음 두 가지 보너스 과제 중 하나를 선택하여 수행한다.

### **보너스 A — FacetGrid 15패널**

`sns.FacetGrid`로 5개 자치구(행) × 3개 room_type(열) = 15패널에 가격 히스토그램을 그리고, 가장 특이한 분포를 가진 패널을 찾는다.

```python
g = sns.FacetGrid(df_under300,
                  row='neighbourhood_group',
                  col='room_type',
                  height=2.5, aspect=1.3,
                  margin_titles=True)
g.map_dataframe(sns.histplot, x='price', bins=20, kde=True)
g.set_axis_labels('가격 ($)', '빈도')
plt.suptitle('자치구 × 유형별 가격 분포 (15패널)', y=1.02)
plt.tight_layout()
plt.show()
```

### **보너스 B — pairplot 다변수 산점도**

4개 숫자 변수의 모든 쌍 산점도를 자치구별 색상으로 분리하여 그린다.

```python
cols_pair = ['price','number_of_reviews',
             'minimum_nights','availability_365']
df_pair = df_under300[cols_pair + ['neighbourhood_group']]

sns.pairplot(df_pair, hue='neighbourhood_group',
             palette='colorblind',
             plot_kws={'alpha':0.3, 's':10},
             diag_kind='kde', height=2.3)
plt.suptitle('주요 숫자 변수 페어플롯', y=1.01)
plt.show()
```

---

> **미션 종료**
>
> 본 미션을 완료하면, 4주차의 `describe()` 통계량과 본 차시의 **시각적 EDA**를 모두 거친 셈이다. 다음 차시부터 시작될 머신러닝 단원에서, 본 미션의 발견들이 **특성 선택·전처리·모델 선택**의 근거로 직접 사용된다. EDA의 가치는 분석에서 끝나지 않고 **모델로 이어진다.**
